# CO-SPY Kaggle Training

Notebook này tự clone repository, cài dependency, kiểm tra GPU, liên kết dataset và huấn luyện. Sau khi import vào Kaggle:

1. Thêm dataset **Co-Spy_v1** bằng **Add Input**.
2. Nếu cần tiếp tục, thêm **Output của version trước** bằng **Add Input**. Notebook tự khôi phục `ckpt` và semantic cache.
3. Giữ **Accelerator = GPU T4 x2** và bật **Internet**.
4. Chọn **Run All**. Notebook tự chạy tuần tự nếu chỉ có một GPU.

Checkpoint và feature cache được giữ trong `/kaggle/working/cospy/ckpt`. Khi time limit, Save Version rồi Add Input output đó vào lần chạy kế tiếp.

In [ ]:
from pathlib import Path
import os
import shutil
from zipfile import ZipFile, BadZipFile
import subprocess

repo_root = Path("/kaggle/working/cospy")

if not (repo_root / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/datnguyentrung/CO-SPY.git",
            str(repo_root),
        ],
        check=True,
    )

repo_ref = os.environ.get("CO_SPY_REPO_REF", "main")
subprocess.run(["git", "fetch", "--depth", "1", "origin", repo_ref], cwd=repo_root, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo_root, check=True)

os.chdir(repo_root)
# Never silently train with an older GitHub checkout. The optimized notebook
# requires these markers; push this repository before publishing the notebook.
required_markers = {
    "main.py": ("--precision", "--profile-batches"),
    "train.py": ("profile_batches", "data wait"),
    "detectors/sd_v1_4/feature_cache.py": ("FeatureCacheInterrupted", ".partial"),
}
missing = [
    f"{path}:{marker}"
    for path, markers in required_markers.items()
    for marker in markers
    if marker not in (repo_root / path).read_text(encoding="utf-8")
]
if missing:
    raise RuntimeError(
        "The GitHub checkout is older than this notebook. Push the optimized "
        "CO-SPY revision, then create a new Kaggle version. Missing: " + ", ".join(missing)
    )
print(f"[SETUP] Working directory: {Path.cwd()} | optimized revision verified", flush=True)

In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys

packages = [
    "diffusers==0.32.2",
    "open-clip-torch==2.31.0",
    "timm==1.0.15",
    "loguru==0.7.3",
    "accelerate==1.0.1",
    "ftfy==6.2.3",
    "tqdm==4.67.1",
    "transformers==4.46.3",
    "safetensors==0.5.3",
]

print("[SETUP] Installing CO-SPY dependencies...", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

expected_versions = {
    "transformers": "4.46.3",
    "safetensors": "0.5.3",
}
for package, expected in expected_versions.items():
    actual = importlib.metadata.version(package)
    if actual != expected:
        raise RuntimeError(
            f"Expected {package}=={expected}, found {package}=={actual}. "
            "Restart the Kaggle session and rerun this cell."
        )

importlib.import_module("diffusers")
importlib.import_module("open_clip")
from diffusers import AutoencoderKL

import os
import torch

worker_count = max(1, (os.cpu_count() or 2) // (2 if torch.cuda.device_count() >= 2 else 1))

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Kaggle, enable Accelerator > GPU T4 x2 "
        "and Internet, then run Restart & Run All."
    )

print("[SETUP] Dependency import check passed.", flush=True)
print(
    f"[SETUP] CUDA GPUs available: {torch.cuda.device_count()}",
    flush=True,
)
print("[SETUP] Dependencies ready.", flush=True)

In [ ]:
from pathlib import Path
import os

import shutil
from zipfile import ZipFile, BadZipFile

input_root = Path("/kaggle/input")
working_root = Path.cwd()
checkpoint_root = working_root / "ckpt"


def shallow_input_directories(root: Path, max_depth: int = 2):
    ignored = {"mscoco2017", "stable-diffusion-v1-4", "train2017", "val2017"}
    result = [root]
    for current, dir_names, _ in os.walk(root, followlinks=False):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)
        dir_names[:] = [name for name in dir_names if name not in ignored]
        if depth >= max_depth:
            dir_names[:] = []
            continue
        for name in dir_names:
            result.append(current_path / name)
    return result


def safe_extract_checkpoint_bundle(archive_path: Path) -> bool:
    try:
        with ZipFile(archive_path) as archive:
            members = [name for name in archive.namelist() if name.startswith("ckpt/")]
            if not members:
                return False
            if any(Path(name).is_absolute() or ".." in Path(name).parts for name in members):
                raise RuntimeError(f"Unsafe checkpoint archive: {archive_path}")
            archive.extractall(working_root, members=members)
        print(f"[RESUME] Extracted checkpoint bundle: {archive_path}", flush=True)
        return True
    except BadZipFile:
        return False


candidate_roots = shallow_input_directories(input_root)
bundle_paths = []
checkpoint_dirs = []
for candidate in candidate_roots:
    direct_checkpoint = candidate / "ckpt"
    if direct_checkpoint.is_dir():
        checkpoint_dirs.append(direct_checkpoint)
    if candidate.is_dir():
        bundle_paths.extend(
            path for path in candidate.glob("*.zip")
            if "checkpoint" in path.name.casefold() or "resume" in path.name.casefold()
        )

for bundle_path in sorted(set(bundle_paths)):
    safe_extract_checkpoint_bundle(bundle_path)

for source_checkpoint in sorted(set(checkpoint_dirs)):
    if source_checkpoint.resolve() == checkpoint_root.resolve():
        continue
    shutil.copytree(source_checkpoint, checkpoint_root, dirs_exist_ok=True)
    print(f"[RESUME] Restored checkpoint tree: {source_checkpoint}", flush=True)

if checkpoint_root.exists():
    restored_files = sum(1 for path in checkpoint_root.rglob("*") if path.is_file())
    print(f"[RESUME] Local checkpoint files ready: {restored_files}", flush=True)
else:
    print("[RESUME] No prior output supplied; starting a new run.", flush=True)

default_dataset_root = Path(
    "/kaggle/input/datasets/datnguyentrung/co-spy-v1"
)
override_root = os.environ.get("CO_SPY_DATASET_ROOT", "").strip()

search_roots = []
for candidate in (
    Path(override_root) if override_root else None,
    default_dataset_root,
    input_root,
):
    if candidate is not None and candidate.is_dir():
        if all(candidate != existing for existing in search_roots):
            search_roots.append(candidate)


def find_directory(search_root, target_name, max_depth=8):
    search_root = Path(search_root)
    target_key = target_name.casefold()
    direct = search_root / target_name
    if direct.is_dir():
        return direct

    for current, dir_names, _ in os.walk(
        search_root,
        followlinks=False,
    ):
        current_path = Path(current)
        depth = len(current_path.relative_to(search_root).parts)
        dir_names.sort(key=str.casefold)
        for dir_name in dir_names:
            if dir_name.casefold() == target_key:
                return current_path / dir_name
        if depth >= max_depth:
            dir_names[:] = []

    raise FileNotFoundError(
        f"Cannot find directory {target_name!r} below {search_root}"
    )


def locate_source_dir(target_name):
    for search_root in search_roots:
        try:
            found = find_directory(search_root, target_name)
        except FileNotFoundError:
            continue
        print(
            f"[DATA] {target_name} source: {found}",
            flush=True,
        )
        return found

    checked = "\n".join(f"  - {root}" for root in search_roots)
    try:
        top_level = "\n".join(
            f"  - {path}" for path in sorted(input_root.iterdir())
        )
    except OSError as exc:
        top_level = f"  <cannot list {input_root}: {exc}>"
    raise FileNotFoundError(
        f"Cannot find {target_name!r} under Kaggle input.\n"
        f"Checked:\n{checked or '  <no input roots found>'}\n"
        f"Top-level {input_root}:\n{top_level}"
    )


real_source_root = locate_source_dir("mscoco2017")
fake_source_root = locate_source_dir("stable-diffusion-v1-4")

print(
    "[DATA] Dataset roots located successfully.",
    flush=True,
)

working_root = Path.cwd()
sd_root = working_root / "data" / "train" / "sd-v1_4"
real_target_root = sd_root / "mscoco2017"
fake_target_root = sd_root / "stable-diffusion-v1-4"
real_target_root.mkdir(parents=True, exist_ok=True)
fake_target_root.mkdir(parents=True, exist_ok=True)


def find_split_dir(source_parent: Path, split: str) -> Path:
    names = (f"{split}2017", split)
    matches = []

    for name in names:
        matches.extend(
            path
            for path in source_parent.rglob(name)
            if path.is_dir()
        )

    if not matches:
        raise FileNotFoundError(
            f"Cannot find {names[0]} or {names[1]} under {source_parent}"
        )

    return sorted(matches)[0]


def link_split(source_parent: Path, target_parent: Path, split: str) -> Path:
    expected = f"{split}2017"
    source_dir = find_split_dir(source_parent, split)
    target = target_parent / expected

    if target.is_symlink():
        target.unlink()
    elif target.is_dir():
        print(f"[DATA] Keeping existing directory: {target}", flush=True)
        return target

    target.symlink_to(source_dir, target_is_directory=True)
    print(
        f"[DATA] Linked {target} -> {source_dir}",
        flush=True,
    )
    return target


linked_paths = {
    "SD train": link_split(
        fake_source_root,
        fake_target_root,
        "train",
    ),
    "SD val": link_split(
        fake_source_root,
        fake_target_root,
        "val",
    ),
    "COCO train": link_split(
        real_source_root,
        real_target_root,
        "train",
    ),
    "COCO val": link_split(
        real_source_root,
        real_target_root,
        "val",
    ),
}

image_extensions = ("*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG", "*.JPEG")

for label, image_dir in linked_paths.items():
    # Do not count a whole split here: recursively stat'ing hundreds of
    # thousands of Kaggle input files is slow and adds no training value.
    first_image = next(
        (path for pattern in image_extensions for path in image_dir.glob(pattern)),
        None,
    )
    if first_image is None:
        raise RuntimeError(f"No directly stored image found in {image_dir}")
    print(f"[DATA] {label:<12} ready | {image_dir}", flush=True)

print(
    "[DATA] Ready. No download or extraction was performed.",
    flush=True,
)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import threading

REPO_ROOT = Path("/kaggle/working/cospy")
PRINT_LOCK = threading.Lock()


def _stream_output(tag, process, log_path):
    with log_path.open("a", encoding="utf-8", buffering=1) as log_file:
        for raw_line in process.stdout:
            line = raw_line.rstrip("\n")
            log_file.write(raw_line)
            log_file.flush()

            with PRINT_LOCK:
                print(f"[{tag}] {line}", flush=True)


def start_job(tag, log_name, args):
    log_path = REPO_ROOT / log_name
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    command = [sys.executable, "-u", "main.py", *args]

    print(f"[{tag}] START: {' '.join(command)}", flush=True)

    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    reader = threading.Thread(
        target=_stream_output,
        args=(tag, process, log_path),
        daemon=True,
    )
    reader.start()
    return process, reader


def run_job(tag, log_name, args):
    process, reader = start_job(tag, log_name, args)
    return_code = process.wait()
    reader.join()

    print(f"[{tag}] EXIT CODE: {return_code}", flush=True)
    if return_code != 0:
        raise RuntimeError(f"{tag} failed with exit code {return_code}")


def choose_profile(tag, base_args, candidates, measured_batches=12):
    """Benchmark candidates and return the fastest successful config."""
    import re

    results = []
    gpu = int(base_args[base_args.index("--gpu") + 1])
    vram_limit_mib = 0.90 * torch.cuda.get_device_properties(gpu).total_memory / (1024 ** 2)
    for candidate in candidates:
        args = [*base_args, "--batch_size", str(candidate["batch_size"]),
                "--num-workers", str(candidate["num_workers"]),
                "--prefetch-factor", str(candidate["prefetch_factor"]),
                "--precision", candidate["precision"],
                "--profile-batches", str(measured_batches), "--no-cache-features",
                "--no-resume"]
        completed = subprocess.run(
            [sys.executable, "-u", "main.py", *args], cwd=REPO_ROOT,
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        profile_log = REPO_ROOT / f"profile_{tag.lower()}_b{candidate['batch_size']}_{candidate['precision']}.log"
        profile_log.write_text(completed.stdout, encoding="utf-8")
        match = re.search(r"samples_per_second=([0-9.]+).*peak_vram_mib=([0-9.]+)", completed.stdout)
        if completed.returncode == 0 and match:
            speed = float(match.group(1))
            peak_vram = float(match.group(2))
            if peak_vram <= vram_limit_mib:
                results.append((speed, candidate))
                print(f"[PROFILE] {tag} {candidate}: {speed:.2f} samples/s | {peak_vram:.0f} MiB", flush=True)
            else:
                print(f"[PROFILE] {tag} rejected {candidate}: {peak_vram:.0f} MiB exceeds 90% VRAM budget", flush=True)
        else:
            print(f"[PROFILE] {tag} skipped {candidate}; see {profile_log.name}", flush=True)
    if not results:
        raise RuntimeError(f"No profile candidate completed for {tag}")
    speed, selected = max(results, key=lambda item: item[0])
    print(f"[PROFILE] Selected {tag}: {selected} at {speed:.2f} samples/s", flush=True)
    return selected

In [ ]:
import torch

# Short real-data benchmarks choose the fastest setting that fits this T4.
# Logs are kept in the Kaggle output for later comparison.
semantic_gpu = 1 if torch.cuda.device_count() >= 2 else 0
profile_workers = [max(1, worker_count // 2), worker_count]
artifact_profile = choose_profile(
    "ART", ["--phase", "train", "--gpu", "0", "--mode", "branch",
            "--branch", "artifact", "--train_dataset", "sd-v1_4"],
    [
        {"batch_size": 16, "num_workers": profile_workers[0], "prefetch_factor": 2, "precision": "fp32"},
        {"batch_size": 16, "num_workers": worker_count, "prefetch_factor": 2, "precision": "auto"},
        {"batch_size": 32, "num_workers": worker_count, "prefetch_factor": 2, "precision": "auto"},
    ],
)
semantic_profile = choose_profile(
    "SEM", ["--phase", "train", "--gpu", str(semantic_gpu), "--mode", "branch",
            "--branch", "semantic", "--train_dataset", "sd-v1_4", "--feat_interp"],
    [
        {"batch_size": 32, "num_workers": profile_workers[0], "prefetch_factor": 2, "precision": "fp32"},
        {"batch_size": 32, "num_workers": worker_count, "prefetch_factor": 2, "precision": "auto"},
        {"batch_size": 64, "num_workers": worker_count, "prefetch_factor": 2, "precision": "auto"},
    ],
)

def train_args(gpu, branch, profile, extra=()):
    return [
        "--phase", "train", "--gpu", str(gpu), "--mode", "branch",
        "--branch", branch, "--train_dataset", "sd-v1_4",
        "--epochs", "5", "--early-stopping-patience", "2",
        "--batch_size", str(profile["batch_size"]),
        "--num-workers", str(profile["num_workers"]),
        "--prefetch-factor", str(profile["prefetch_factor"]),
        "--precision", profile["precision"], *extra,
    ]

artifact_args = train_args(0, "artifact", artifact_profile)
semantic_args = train_args(semantic_gpu, "semantic", semantic_profile, ("--feat_interp",))

if torch.cuda.device_count() >= 2:
    print(
        "[TRAIN] Running Artifact on GPU 0 and Semantic on GPU 1.",
        flush=True,
    )
    artifact_process, artifact_reader = start_job(
        "ART",
        "train_artifact.log",
        artifact_args,
    )
    semantic_process, semantic_reader = start_job(
        "SEM",
        "train_semantic.log",
        semantic_args,
    )

    artifact_code = artifact_process.wait()
    semantic_code = semantic_process.wait()
    artifact_reader.join()
    semantic_reader.join()

    print(
        f"[TRAIN] Artifact exit={artifact_code} | "
        f"Semantic exit={semantic_code}",
        flush=True,
    )
    if artifact_code != 0 or semantic_code != 0:
        raise RuntimeError(
            "At least one branch failed; check the logs above."
        )
else:
    print(
        "[TRAIN] One GPU detected; running the branches sequentially.",
        flush=True,
    )
    run_job("ART", "train_artifact.log", artifact_args)
    run_job("SEM", "train_semantic.log", semantic_args)

In [ ]:
npr_profile = choose_profile(
    "NPR", ["--phase", "train", "--gpu", "0", "--mode", "branch",
            "--branch", "npr", "--train_dataset", "sd-v1_4"],
    [
        {"batch_size": 32, "num_workers": worker_count, "prefetch_factor": 2, "precision": "fp32"},
        {"batch_size": 32, "num_workers": worker_count, "prefetch_factor": 2, "precision": "auto"},
        {"batch_size": 64, "num_workers": worker_count, "prefetch_factor": 2, "precision": "auto"},
    ],
)
run_job(
    "NPR", "train_npr.log",
    train_args(0, "npr", npr_profile),
)

In [ ]:
run_job(
    "FUS3",
    "train_fusion_v3.log",
    [
        "--phase", "train",
        "--gpu", "0",
        "--mode", "fusion",
        "--train_dataset", "sd-v1_4",
        "--epochs", "5",
        "--early-stopping-patience", "2",
        "--fusion_head", "v3",
        "--artifact_views", "1",
        "--num-workers", str(worker_count),
        "--prefetch-factor", "2",
        "--precision", "auto",
    ],
)

In [ ]:
# Fusion v3 is trained in the preceding cell. Kept as a no-op so older
# links to this cell remain harmless when using Run All.
print("[PIPELINE] Fusion v2 is intentionally skipped; Fusion v3 already completed.", flush=True)

In [ ]:
# Resume is automatic in each train command; no second Artifact pass is run.
print("[PIPELINE] Artifact resume is handled by its checkpoint manager.", flush=True)

In [ ]:
from pathlib import Path

checkpoint_root = Path("ckpt")
checkpoint_files = sorted(
    path
    for path in checkpoint_root.rglob("*")
    if path.is_file()
    and (
        path.name.startswith("checkpoint_")
        or path.name == "training_state.json"
    )
)

for path in checkpoint_files:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"[CKPT] {path} | {size_mb:.2f} MB", flush=True)

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

checkpoint_root = Path("ckpt")
archive_path = Path("cospy_resume_bundle.zip")

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in checkpoint_root.rglob("*"):
        if not path.is_file():
            continue
        archive.write(path, path.as_posix())

print(
    f"[PACK] Created resumable bundle {archive_path} (includes feature cache) "
    f"({archive_path.stat().st_size / (1024 ** 3):.2f} GB)",
    flush=True,
)